# Boucle 3 : Programmation Lineaire

Ce notebook introduit la **programmation lineaire** (PL) et la **programmation lineaire en nombres entiers** (PLNE), deux outils fondamentaux de la recherche operationnelle. L'objectif est de comprendre comment formuler un probleme d'optimisation sous contraintes, le resoudre avec Python, et faire le lien avec le projet ADEME.

Le notebook est concu pour etre execute de haut en bas : plusieurs cellules reutilisent des variables definies dans les sections precedentes.

## 1. Introduction a la Programmation Lineaire

### 1.1 Qu'est-ce que la programmation lineaire ?

La **programmation lineaire** est une methode d'optimisation mathematique dans laquelle :
- La **fonction objectif** (ce qu'on cherche a maximiser ou minimiser) est lineaire
- Les **contraintes** sont des inegalites ou egalites lineaires
- Les **variables de decision** sont continues (nombres reels positifs)

Formellement, un probleme de PL s'ecrit :

$$\min_{x} \quad c^T x$$
$$\text{sous contraintes} \quad Ax \leq b$$
$$x \geq 0$$

### 1.2 Domaines d'application

La PL est omnipresente dans l'industrie et la logistique :
- **Logistique** : planification de tournees, gestion de stocks
- **Production** : allocation optimale des ressources, planification de fabrication
- **Transport** : affectation de vehicules, minimisation des couts de livraison
- **Finance** : optimisation de portefeuille, gestion des risques
- **Energie** : gestion de reseaux electriques, planification de production

### 1.3 Lien avec le projet ADEME

Dans le projet ADEME, nous cherchons a optimiser des tournees de collecte en minimisant les distances parcourues (ou les emissions de CO2), tout en respectant des contraintes operationnelles : capacite des vehicules, fenetres horaires, nombre de sites a visiter. La programmation lineaire en nombres entiers est l'outil mathematique exact pour formuler et resoudre ce type de probleme.

## 2. Formulation d'un probleme lineaire

### 2.1 Les trois ingredients

Tout probleme de PL se decompose en trois elements :

1. **Variables de decision** : les quantites que l'on cherche a determiner
2. **Fonction objectif** : le critere a optimiser (maximiser le profit, minimiser le cout...)
3. **Contraintes** : les limites imposees par le probleme reel

### 2.2 Exemple concret : l'usine de meubles

Une usine fabrique deux types de produits : des **tables** et des **chaises**.

| Ressource | Table | Chaise | Disponible |
|-----------|-------|--------|------------|
| Bois (kg) | 5 | 2 | 180 |
| Main d'oeuvre (h) | 3 | 4 | 200 |
| Vernis (L) | 1 | 1 | 50 |

Le profit est de **60 euros** par table et **40 euros** par chaise.

### 2.3 Formulation mathematique

**Variables de decision :**
- $x_1$ : nombre de tables produites
- $x_2$ : nombre de chaises produites

**Fonction objectif (maximiser le profit) :**
$$\max \quad Z = 60x_1 + 40x_2$$

**Contraintes :**
$$5x_1 + 2x_2 \leq 180 \quad \text{(bois)}$$
$$3x_1 + 4x_2 \leq 200 \quad \text{(main d'oeuvre)}$$
$$x_1 + x_2 \leq 50 \quad \text{(vernis)}$$
$$x_1, x_2 \geq 0$$

## 3. Resolution graphique (2 variables)

Lorsque le probleme ne comporte que deux variables, on peut visualiser la **region admissible** (l'ensemble des solutions respectant toutes les contraintes) et trouver l'optimum graphiquement.

Un resultat fondamental de la PL : **l'optimum se trouve toujours en un sommet du polyedre** defini par les contraintes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from scipy.optimize import linprog


def tracer_region_admissible():
    """Trace la region admissible et la solution optimale du probleme de l'usine."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))

    # Grille pour x1
    x1 = np.linspace(0, 50, 400)

    # Contraintes reecrites en x2 = f(x1)
    # Bois : 5*x1 + 2*x2 <= 180  =>  x2 <= (180 - 5*x1) / 2
    x2_bois = (180 - 5 * x1) / 2

    # Main d'oeuvre : 3*x1 + 4*x2 <= 200  =>  x2 <= (200 - 3*x1) / 4
    x2_mo = (200 - 3 * x1) / 4

    # Vernis : x1 + x2 <= 50  =>  x2 <= 50 - x1
    x2_vernis = 50 - x1

    # Tracer les droites de contraintes
    ax.plot(x1, x2_bois, label="Bois : $5x_1 + 2x_2 \\leq 180$", color="blue")
    ax.plot(x1, x2_mo, label="Main d'oeuvre : $3x_1 + 4x_2 \\leq 200$", color="red")
    ax.plot(x1, x2_vernis, label="Vernis : $x_1 + x_2 \\leq 50$", color="green")

    # Region admissible : minimum des contraintes, tronquee a 0
    x2_min = np.minimum(np.minimum(x2_bois, x2_mo), x2_vernis)
    x2_min = np.maximum(x2_min, 0)

    ax.fill_between(x1, 0, x2_min, where=(x2_min >= 0) & (x1 >= 0),
                    alpha=0.2, color="yellow", label="Region admissible")

    # Sommets du polyedre (calcules a la main pour cet exemple)
    sommets = [
        (0, 0),
        (36, 0),       # Bois : 5*36 = 180
        (20, 40),      # Intersection bois et vernis : 5x1+2x2=180 et x1+x2=50
        (0, 50),       # x1=0, x2=50 (vernis)
    ]

    # Verifier quel sommet est le vrai optimum
    # On recalcule proprement les intersections
    # Bois & MO : 5x1+2x2=180, 3x1+4x2=200 => x1=20, x2=40? non
    # 5*20+2*40=100+80=180 ok, 3*20+4*40=60+160=220 > 200 non!
    # Bois & MO : 10x1+4x2=360, 3x1+4x2=200 => 7x1=160 => x1=160/7~22.86, x2=(180-5*22.86)/2~32.86
    # Bois & Vernis : 5x1+2(50-x1)=180 => 3x1=80 => x1=80/3~26.67, x2=50-26.67=23.33
    # MO & Vernis : 3x1+4(50-x1)=200 => -x1=-0 => x1=0, x2=50

    sommets_corriges = [
        (0, 0),
        (36, 0),             # Bois & axe x1
        (160 / 7, (180 - 5 * 160 / 7) / 2),   # Bois & MO
        (80 / 3, 50 - 80 / 3),                  # Bois & Vernis
        (0, 50),             # Vernis & axe x2
    ]

    # Filtrer les sommets admissibles
    sommets_admissibles = []
    for (sx1, sx2) in sommets_corriges:
        if (sx1 >= -0.01 and sx2 >= -0.01 and
            5 * sx1 + 2 * sx2 <= 180.01 and
            3 * sx1 + 4 * sx2 <= 200.01 and
            sx1 + sx2 <= 50.01):
            sommets_admissibles.append((sx1, sx2))

    # Evaluer la fonction objectif en chaque sommet admissible
    meilleur_z = -1
    meilleur_sommet = None
    print("Evaluation de la fonction objectif aux sommets :")
    for (sx1, sx2) in sommets_admissibles:
        z = 60 * sx1 + 40 * sx2
        print(f"  ({sx1:.2f}, {sx2:.2f}) -> Z = {z:.2f}")
        ax.plot(sx1, sx2, "ko", markersize=5)
        if z > meilleur_z:
            meilleur_z = z
            meilleur_sommet = (sx1, sx2)

    # Marquer l'optimum
    ax.plot(meilleur_sommet[0], meilleur_sommet[1], "r*", markersize=20,
            label=f"Optimum : Z = {meilleur_z:.2f}")
    ax.annotate(f"  ({meilleur_sommet[0]:.1f}, {meilleur_sommet[1]:.1f})",
                xy=meilleur_sommet, fontsize=11, color="red")

    # Courbes de niveau de la fonction objectif
    for z_val in [500, 1000, 1500, 2000]:
        x2_iso = (z_val - 60 * x1) / 40
        ax.plot(x1, x2_iso, "--", color="gray", alpha=0.3, linewidth=0.8)

    ax.set_xlim(-2, 55)
    ax.set_ylim(-2, 100)
    ax.set_xlabel("$x_1$ (tables)", fontsize=12)
    ax.set_ylabel("$x_2$ (chaises)", fontsize=12)
    ax.set_title("Resolution graphique du probleme de l'usine", fontsize=14)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


tracer_region_admissible()

On observe que la solution optimale se trouve bien a un sommet du polyedre. C'est une propriete fondamentale de la PL : l'algorithme du **simplexe** exploite cette propriete en se deplacant de sommet en sommet jusqu'a trouver l'optimum.

## 4. Introduction a PuLP

**PuLP** est une bibliotheque Python qui permet de formuler et resoudre des problemes de programmation lineaire de facon claire et lisible. Elle fait appel a des solveurs (CBC, GLPK, Gurobi...) pour effectuer la resolution.

### 4.1 Installation

In [ ]:
# Installation de PuLP (a executer une seule fois)
# Le solveur CBC est inclus par defaut avec PuLP
%pip install pulp -q

### 4.2 Syntaxe de base

Les etapes pour resoudre un probleme avec PuLP :

1. Creer le probleme (`LpProblem`)
2. Definir les variables de decision (`LpVariable`)
3. Ajouter la fonction objectif
4. Ajouter les contraintes
5. Resoudre (`solve()`)
6. Lire les resultats

### 4.3 Resolution de l'exemple de l'usine

In [ ]:
import pulp


def resoudre_usine_pl():
    """
    Resout le probleme de l'usine de meubles en programmation lineaire continue.

    Retourne le modele resolu.
    """
    # 1. Creer le probleme (maximisation)
    probleme = pulp.LpProblem("Usine_de_meubles", pulp.LpMaximize)

    # 2. Variables de decision (continues, positives)
    x1 = pulp.LpVariable("tables", lowBound=0, cat="Continuous")
    x2 = pulp.LpVariable("chaises", lowBound=0, cat="Continuous")

    # 3. Fonction objectif : maximiser le profit
    probleme += 60 * x1 + 40 * x2, "Profit_total"

    # 4. Contraintes
    probleme += 5 * x1 + 2 * x2 <= 180, "Contrainte_bois"
    probleme += 3 * x1 + 4 * x2 <= 200, "Contrainte_main_oeuvre"
    probleme += x1 + x2 <= 50, "Contrainte_vernis"

    # 5. Resoudre
    probleme.solve(pulp.PULP_CBC_CMD(msg=0))

    # 6. Afficher les resultats
    print("=" * 50)
    print("RESOLUTION PL CONTINUE")
    print("=" * 50)
    print(f"Statut : {pulp.LpStatus[probleme.status]}")
    print(f"Tables (x1) : {x1.varValue:.2f}")
    print(f"Chaises (x2) : {x2.varValue:.2f}")
    print(f"Profit maximal : {pulp.value(probleme.objective):.2f} euros")
    print("=" * 50)

    return probleme


modele_pl = resoudre_usine_pl()

On retrouve bien la solution trouvee graphiquement. Remarquez que les valeurs des variables ne sont pas necessairement entieres : en PL continue, on peut produire des fractions de produit. Ce n'est pas toujours realiste...

## 5. Programmation Lineaire en Nombres Entiers (PLNE)

### 5.1 Pourquoi les entiers sont importants ?

Dans de nombreux problemes reels, les variables de decision doivent etre entieres :
- On ne peut pas envoyer **0.7 camion** sur une route
- On ne peut pas construire **2.3 entrepots**
- On ne peut pas affecter **0.5 employe** a une tache

La **programmation lineaire en nombres entiers** (PLNE, ou ILP en anglais) ajoute la contrainte $x \in \mathbb{Z}^+$ aux variables.

### 5.2 Impact sur la complexite

La PL continue se resout en temps polynomial (algorithme du simplexe en pratique, points interieurs en theorie). En revanche, la PLNE est un probleme **NP-difficile** : le temps de resolution peut croitre exponentiellement avec la taille du probleme.

Les solveurs modernes (CBC, Gurobi, CPLEX) utilisent des techniques avancees : *branch and bound*, *branch and cut*, *plans coupants*, pour resoudre efficacement de nombreuses instances en pratique.

### 5.3 Comparaison PL vs PLNE sur l'exemple de l'usine

In [ ]:
def resoudre_usine_plne():
    """
    Resout le probleme de l'usine de meubles en nombres entiers.

    Retourne le modele resolu.
    """
    probleme = pulp.LpProblem("Usine_de_meubles_PLNE", pulp.LpMaximize)

    # Variables entieres cette fois
    x1 = pulp.LpVariable("tables", lowBound=0, cat="Integer")
    x2 = pulp.LpVariable("chaises", lowBound=0, cat="Integer")

    # Fonction objectif
    probleme += 60 * x1 + 40 * x2, "Profit_total"

    # Contraintes (identiques)
    probleme += 5 * x1 + 2 * x2 <= 180, "Contrainte_bois"
    probleme += 3 * x1 + 4 * x2 <= 200, "Contrainte_main_oeuvre"
    probleme += x1 + x2 <= 50, "Contrainte_vernis"

    probleme.solve(pulp.PULP_CBC_CMD(msg=0))

    print("=" * 50)
    print("RESOLUTION PLNE (NOMBRES ENTIERS)")
    print("=" * 50)
    print(f"Statut : {pulp.LpStatus[probleme.status]}")
    print(f"Tables (x1) : {int(x1.varValue)}")
    print(f"Chaises (x2) : {int(x2.varValue)}")
    print(f"Profit maximal : {pulp.value(probleme.objective):.2f} euros")
    print("=" * 50)

    return probleme


modele_plne = resoudre_usine_plne()

**Observation importante :** la solution en nombres entiers peut differer de l'arrondi de la solution continue. Arrondir la solution de la PL continue ne donne pas toujours une solution admissible ou optimale pour le probleme en nombres entiers.

## 6. Application au probleme du voyageur de commerce (TSP)

### 6.1 Rappel du probleme

Le TSP consiste a trouver la tournee de cout minimal passant par chaque ville exactement une fois et revenant au point de depart. C'est un probleme central du projet ADEME (optimisation de tournees de collecte).

### 6.2 Formulation PLNE du TSP

Soit $n$ le nombre de villes et $c_{ij}$ le cout (distance) pour aller de la ville $i$ a la ville $j$.

**Variables de decision :**
$$x_{ij} \in \{0, 1\} \quad \forall i, j \in \{0, ..., n-1\}, \; i \neq j$$

$x_{ij} = 1$ si l'on emprunte l'arc de $i$ vers $j$ dans la tournee, 0 sinon.

**Fonction objectif :**
$$\min \sum_{i} \sum_{j \neq i} c_{ij} \cdot x_{ij}$$

**Contraintes de degre** (chaque ville est visitee exactement une fois) :
$$\sum_{j \neq i} x_{ij} = 1 \quad \forall i \quad \text{(une seule sortie par ville)}$$
$$\sum_{i \neq j} x_{ij} = 1 \quad \forall j \quad \text{(une seule entree par ville)}$$

### 6.3 Elimination des sous-tours (MTZ)

Les contraintes de degre seules ne suffisent pas : elles autorisent des **sous-tours** (plusieurs petites boucles au lieu d'une seule grande tournee). Les contraintes de **Miller-Tucker-Zemlin** (MTZ) eliminent les sous-tours a l'aide de variables auxiliaires $u_i$ representant l'ordre de visite :

$$u_i - u_j + n \cdot x_{ij} \leq n - 1 \quad \forall i, j \in \{1, ..., n-1\}, \; i \neq j$$
$$1 \leq u_i \leq n - 1 \quad \forall i \in \{1, ..., n-1\}$$

L'idee : si $x_{ij} = 1$ (on va de $i$ a $j$), alors $u_j \geq u_i + 1$, ce qui impose un ordre de passage et interdit les sous-tours.

### 6.4 Implementation avec PuLP

In [ ]:
import math
import random

# Generation d'un petit exemple a 5 villes
random.seed(42)

NB_VILLES = 5
NOMS_VILLES = [f"V{i}" for i in range(NB_VILLES)]

# Coordonnees aleatoires
COORDS = {nom: (random.randint(0, 100), random.randint(0, 100)) for nom in NOMS_VILLES}

# Matrice des distances
DISTANCES = {}
for i, vi in enumerate(NOMS_VILLES):
    for j, vj in enumerate(NOMS_VILLES):
        if i != j:
            xi, yi = COORDS[vi]
            xj, yj = COORDS[vj]
            DISTANCES[(vi, vj)] = round(math.sqrt((xj - xi) ** 2 + (yj - yi) ** 2), 2)

print("Coordonnees des villes :")
for nom, (x, y) in COORDS.items():
    print(f"  {nom} : ({x}, {y})")

print(f"\nNombre d'arcs : {len(DISTANCES)}")

In [ ]:
def resoudre_tsp_mtz(noms_villes, distances):
    """
    Resout le TSP avec la formulation MTZ en PLNE.

    Parametres :
        noms_villes : liste des noms de villes
        distances : dictionnaire {(vi, vj): distance}

    Retourne la tournee optimale et sa distance.
    """
    n = len(noms_villes)

    # Creer le probleme
    probleme = pulp.LpProblem("TSP_MTZ", pulp.LpMinimize)

    # Variables de decision : x[i][j] = 1 si on va de i a j
    x = {}
    for i in noms_villes:
        for j in noms_villes:
            if i != j:
                x[(i, j)] = pulp.LpVariable(f"x_{i}_{j}", cat="Binary")

    # Variables auxiliaires MTZ : ordre de visite (sauf ville 0)
    u = {}
    for i in noms_villes[1:]:
        u[i] = pulp.LpVariable(f"u_{i}", lowBound=1, upBound=n - 1, cat="Integer")

    # Fonction objectif : minimiser la distance totale
    probleme += pulp.lpSum(
        distances[(i, j)] * x[(i, j)]
        for i in noms_villes
        for j in noms_villes
        if i != j
    ), "Distance_totale"

    # Contrainte : exactement une sortie par ville
    for i in noms_villes:
        probleme += (
            pulp.lpSum(x[(i, j)] for j in noms_villes if j != i) == 1,
            f"Sortie_{i}"
        )

    # Contrainte : exactement une entree par ville
    for j in noms_villes:
        probleme += (
            pulp.lpSum(x[(i, j)] for i in noms_villes if i != j) == 1,
            f"Entree_{j}"
        )

    # Contraintes MTZ d'elimination des sous-tours
    for i in noms_villes[1:]:
        for j in noms_villes[1:]:
            if i != j:
                probleme += (
                    u[i] - u[j] + n * x[(i, j)] <= n - 1,
                    f"MTZ_{i}_{j}"
                )

    # Resoudre
    probleme.solve(pulp.PULP_CBC_CMD(msg=0))

    # Extraire la tournee
    tournee = [noms_villes[0]]
    ville_courante = noms_villes[0]

    for _ in range(n):
        for j in noms_villes:
            if j != ville_courante and x[(ville_courante, j)].varValue > 0.5:
                tournee.append(j)
                ville_courante = j
                break

    distance_totale = pulp.value(probleme.objective)

    return tournee, round(distance_totale, 2)


tournee_opt, dist_opt = resoudre_tsp_mtz(NOMS_VILLES, DISTANCES)

print("=" * 50)
print("SOLUTION OPTIMALE DU TSP (MTZ)")
print("=" * 50)
print(f"Tournee : {' -> '.join(tournee_opt)}")
print(f"Distance totale : {dist_opt}")
print("=" * 50)

In [ ]:
def visualiser_tournee(coords, tournee, titre="Tournee optimale (TSP)"):
    """Visualise la tournee sur un graphique 2D."""
    fig, ax = plt.subplots(figsize=(8, 8))

    # Tracer les arcs de la tournee
    for k in range(len(tournee) - 1):
        vi, vj = tournee[k], tournee[k + 1]
        xi, yi = coords[vi]
        xj, yj = coords[vj]
        ax.annotate("",
                    xy=(xj, yj), xytext=(xi, yi),
                    arrowprops=dict(arrowstyle="->", color="steelblue", lw=2))

    # Tracer les villes
    for nom, (x, y) in coords.items():
        ax.plot(x, y, "ro", markersize=12)
        ax.annotate(f"  {nom}", xy=(x, y), fontsize=12, fontweight="bold")

    ax.set_title(titre, fontsize=14)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


visualiser_tournee(COORDS, tournee_opt)

## 7. Limites et ouverture

### 7.1 Limites de la programmation lineaire

La PL continue est efficace et se resout en temps polynomial. Cependant :
- Elle ne gere pas les decisions discretes (oui/non, entier)
- La solution continue n'est pas toujours exploitable en pratique

### 7.2 Limites de la PLNE

La PLNE permet de modeliser des problemes combinatoires complexes, mais :
- La resolution est **NP-difficile** dans le cas general
- Le temps de resolution peut exploser avec la taille de l'instance
- Le TSP avec la formulation MTZ devient tres lent au-dela de **20-30 villes**
- Pour des instances industrielles (centaines ou milliers de villes), les solveurs exacts ne suffisent plus

### 7.3 Vers les metaheuristiques (Boucle 4)

Quand la PLNE atteint ses limites, on se tourne vers des methodes approchees :
- **Heuristiques constructives** : plus proche voisin, insertion (vues en Boucle 1)
- **Recherche locale** : 2-opt, 3-opt, Or-opt
- **Metaheuristiques** : recuit simule, algorithmes genetiques, colonies de fourmis
- **Matheuristiques** : combinaison de methodes exactes et heuristiques

Ces approches seront etudiees dans la **Boucle 4** et appliquees au projet ADEME pour traiter des instances de taille realiste.

| Methode | Qualite de la solution | Temps de calcul | Taille max |
|---------|----------------------|-----------------|------------|
| PL continue | Relaxation (borne) | Polynomial | Tres grande |
| PLNE exacte | Optimale | Exponentiel (pire cas) | ~30 villes (TSP) |
| Heuristiques | Approchee | Rapide | Tres grande |
| Metaheuristiques | Bonne a tres bonne | Configurable | Tres grande |

## 8. Exercices

### Exercice 1 : Probleme de regime alimentaire

Un dieteticien souhaite composer un repas au **cout minimal** a partir de 4 aliments, tout en respectant des apports nutritionnels minimaux.

| Aliment | Proteines (g) | Glucides (g) | Lipides (g) | Cout (euros/portion) |
|---------|--------------|-------------|-------------|---------------------|
| Poulet | 25 | 0 | 5 | 3.0 |
| Riz | 5 | 45 | 1 | 1.0 |
| Legumes | 3 | 10 | 0 | 2.0 |
| Fromage | 15 | 0 | 12 | 2.5 |

**Contraintes nutritionnelles minimales :**
- Proteines : au moins 50 g
- Glucides : au moins 60 g
- Lipides : au plus 30 g

Formulez et resolvez ce probleme avec PuLP. Les variables representent le nombre de portions de chaque aliment (valeurs continues positives).

In [ ]:
def resoudre_regime():
    """
    Resout le probleme de regime alimentaire (minimiser le cout
    tout en respectant les contraintes nutritionnelles).

    A COMPLETER par les etudiants.
    """
    pass


# resoudre_regime()

### Exercice 2 : Probleme de transport

Une entreprise dispose de **3 entrepots** et doit livrer **4 clients**. Chaque entrepot a une capacite limitee et chaque client a une demande a satisfaire.

**Capacites des entrepots :**
- E1 : 300 unites
- E2 : 250 unites
- E3 : 450 unites

**Demandes des clients :**
- C1 : 200 unites
- C2 : 150 unites
- C3 : 300 unites
- C4 : 250 unites

**Couts de transport (euros/unite) :**

| | C1 | C2 | C3 | C4 |
|---|---|---|---|---|
| E1 | 8 | 6 | 10 | 9 |
| E2 | 9 | 12 | 7 | 7 |
| E3 | 14 | 9 | 16 | 5 |

Formulez et resolvez ce probleme avec PuLP : minimiser le cout total de transport en satisfaisant toutes les demandes sans depasser les capacites.

In [ ]:
def resoudre_transport():
    """
    Resout le probleme de transport (minimiser le cout total
    de livraison entre entrepots et clients).

    A COMPLETER par les etudiants.
    """
    pass


# resoudre_transport()

### Exercice 3 : TSP avec fenetres temporelles

Reprenez la formulation MTZ du TSP (section 6) et ajoutez des **fenetres temporelles**. Chaque ville $i$ doit etre visitee entre un instant $a_i$ (debut) et $b_i$ (fin). Le temps de trajet entre les villes $i$ et $j$ est proportionnel a la distance : $t_{ij} = d_{ij} / v$ avec $v = 10$ (vitesse constante).

**Donnees supplementaires :**

| Ville | Debut ($a_i$) | Fin ($b_i$) |
|-------|--------------|-------------|
| V0 | 0 | 1000 |
| V1 | 10 | 50 |
| V2 | 20 | 80 |
| V3 | 5 | 60 |
| V4 | 30 | 90 |

**Variables supplementaires :**
- $t_i$ : instant d'arrivee a la ville $i$

**Contraintes supplementaires :**
- $a_i \leq t_i \leq b_i \quad \forall i$
- $t_i + t_{ij} - M(1 - x_{ij}) \leq t_j \quad \forall i \neq j$ (si on va de $i$ a $j$, on arrive apres le temps de trajet)

Ou $M$ est une grande constante (par exemple $M = 1000$).

Implementez cette extension et comparez la solution avec celle du TSP sans fenetres temporelles.

In [ ]:
def resoudre_tsp_fenetres(noms_villes, distances, fenetres, vitesse=10):
    """
    Resout le TSP avec fenetres temporelles en PLNE.

    Parametres :
        noms_villes : liste des noms de villes
        distances : dictionnaire {(vi, vj): distance}
        fenetres : dictionnaire {ville: (debut, fin)}
        vitesse : vitesse constante pour calculer les temps de trajet

    A COMPLETER par les etudiants.
    """
    pass


# Donnees des fenetres temporelles
# FENETRES = {
#     "V0": (0, 1000),
#     "V1": (10, 50),
#     "V2": (20, 80),
#     "V3": (5, 60),
#     "V4": (30, 90),
# }

# tournee_tw, dist_tw = resoudre_tsp_fenetres(NOMS_VILLES, DISTANCES, FENETRES)
# print(f"Tournee avec fenetres : {' -> '.join(tournee_tw)}")
# print(f"Distance : {dist_tw}")